# Advanced Features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/nlp-learning-journey/blob/main/examples/spaCy-Linguistic/05-spaCy-Linguistic-Advanced-Features.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/vuhung16au/nlp-learning-journey/blob/main/examples/spaCy-Linguistic/05-spaCy-Linguistic-Advanced-Features.ipynb)
[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/vuhung16au/nlp-learning-journey/blob/main/examples/spaCy-Linguistic/05-spaCy-Linguistic-Advanced-Features.ipynb)


## Learning Objectives
By the end of this notebook, you will be able to:
- Work with word vectors and similarity
- Customize tokenization
- Modify spaCy pipelines
- Create custom components

## Table of Contents
1. [Word Vectors and Similarity](#word-vectors-and-similarity)
2. [Custom Tokenization](#custom-tokenization)
3. [Pipeline Customization](#pipeline-customization)
4. [Practice Exercises](#practice-exercises)
5. [Summary and Key Takeaways](#summary-and-key-takeaways)


In [ ]:
# Environment Detection and Setup
import sys
import subprocess
import os

# Detect the runtime environment
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

print(f"Environment detected:")
print(f"  - Local: {IS_LOCAL}")
print(f"  - Google Colab: {IS_COLAB}")
print(f"  - Kaggle: {IS_KAGGLE}")

In [ ]:
# Platform-specific spaCy installation and setup
if IS_COLAB:
    print("\nSetting up Google Colab environment...")
    !pip install -q spacy
    !python -m spacy download en_core_web_sm
elif IS_KAGGLE:
    print("\nSetting up Kaggle environment...")
    # Kaggle usually has spaCy pre-installed
    !python -m spacy download en_core_web_sm
else:
    print("\nSetting up local environment...")
    # For local environment, packages should be installed via requirements.txt
    # Verify spaCy model is available
    try:
        import spacy
        nlp = spacy.load("en_core_web_sm")
        print("✓ spaCy and en_core_web_sm model are available")
    except:
        print("⚠ Please run: python -m spacy download en_core_web_sm")

In [1]:
import spacy
from spacy import displacy
import numpy as np

# Load the English language model
nlp = spacy.load("en_core_web_sm")


OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

## 1. Word Vectors and Similarity

### Understanding Word Embeddings

Word vectors (embeddings) are numerical representations of words that capture semantic meaning. spaCy provides pre-trained word vectors for similarity calculations.


In [ ]:
# Check if the model has word vectors
print("Model has vectors:", nlp.vocab.vectors.size > 0)
print("Vector dimensions:", nlp.vocab.vectors.shape)

# Load a model with vectors for better similarity calculations
# Note: You might want to use en_core_web_md for better vector quality
try:
    nlp_vectors = spacy.load("en_core_web_md")
    print("Medium model loaded with vectors:", nlp_vectors.vocab.vectors.size > 0)
except OSError:
    print("Medium model not available. Using small model for demonstration.")
    nlp_vectors = nlp


In [ ]:
# Computing similarity between words
text1 = "I love programming in Python"
text2 = "I enjoy coding in Python"
doc1 = nlp_vectors(text1)
doc2 = nlp_vectors(text2)

print("Text Similarity:")
print("=" * 20)
print(f"Text 1: {text1}")
print(f"Text 2: {text2}")
print(f"Similarity: {doc1.similarity(doc2):.3f}")

# Word-level similarity
print("\nWord Similarities:")
print("-" * 20)
words = ["cat", "dog", "car", "house", "computer"]
for word in words:
    if word in nlp_vectors.vocab:
        similarity = nlp_vectors("cat").similarity(nlp_vectors(word))
        print(f"cat vs {word}: {similarity:.3f}")


### Limitations and Best Practices

**Important considerations:**
- Similarity scores are only meaningful when vectors are available
- Small models may not have vectors (use medium/large models)
- Similarity is based on training data and may not reflect current usage
- Context matters - word similarity ≠ sentence similarity


## 2. Custom Tokenization

### Adding Special Case Tokenization Rules

Most domains have specific tokenization needs. spaCy allows you to add custom rules for special cases.


In [ ]:
# Example: Adding custom tokenization rules
from spacy.symbols import ORTH, NORM

# Create a custom nlp object
nlp_custom = spacy.load("en_core_web_sm")

# Add special case for "don't" to be split as "do" + "n't"
nlp_custom.tokenizer.add_special_case("don't", [
    {ORTH: "do", NORM: "do"},
    {ORTH: "n't", NORM: "not"}
])

# Add special case for "U.S.A." to be kept as one token
nlp_custom.tokenizer.add_special_case("U.S.A.", [
    {ORTH: "U.S.A.", NORM: "USA"}
])

# Test the custom tokenization
text = "I don't think the U.S.A. will change."
doc = nlp_custom(text)

print("Custom Tokenization:")
print("=" * 25)
for token in doc:
    print(f"'{token.text}' -> norm: '{token.norm_}'")


### Custom Tokenizer Patterns

For more complex tokenization needs, you can create custom patterns:


In [ ]:
# Example: Custom tokenizer for scientific text
import re
from spacy.tokenizer import Tokenizer
from spacy.util import compile_infix_regex

# Create custom tokenizer for scientific notation
def custom_tokenizer(nlp):
    # Custom infix patterns
    infixes = (
        nlp.Defaults.infixes + 
        [r"(?<=[0-9])[+\-\*^](?=[0-9])",  # Math operators
        r"(?<=[0-9])[eE](?=[+-]?[0-9])"]  # Scientific notation
    )
    
    # Compile the infix regex
    infix_regex = compile_infix_regex(infixes)
    
    # Create tokenizer with custom infix regex
    return Tokenizer(nlp.vocab, infix_finditer=infix_regex.finditer)

# Apply custom tokenizer
nlp_scientific = spacy.load("en_core_web_sm")
nlp_scientific.tokenizer = custom_tokenizer(nlp_scientific)

# Test with scientific text
scientific_text = "The value is 1.5e-10 and the result is 2.3e+5."
doc = nlp_scientific(scientific_text)

print("Scientific Tokenization:")
print("=" * 30)
for token in doc:
    print(f"'{token.text}'")


## 3. Pipeline Customization

### Understanding spaCy's Pipeline

spaCy processes text through a pipeline of components. You can customize this pipeline by adding, removing, or modifying components.


In [ ]:
# Examine the current pipeline
print("Current pipeline components:")
print("=" * 35)
for name, component in nlp.pipeline:
    print(f"- {name}: {component}")

# Create a minimal pipeline (only tokenizer)
from spacy.lang.en import English

# Create a blank English model
nlp_minimal = English()

print(f"\nMinimal pipeline: {nlp_minimal.pipe_names}")

# Add components to the pipeline
nlp_minimal.add_pipe("tagger")
nlp_minimal.add_pipe("parser")
nlp_minimal.add_pipe("ner")

print(f"After adding components: {nlp_minimal.pipe_names}")


### Adding/Removing Components

You can customize the pipeline by adding or removing components:


In [ ]:
# Create a custom pipeline
from spacy.language import Language
from spacy.tokens import Token

# Register the custom attribute
Token.set_extension("is_custom", default=False)

# Register the custom component
@Language.component("custom_component")
def custom_component(doc):
    # Add a custom attribute to each token
    for token in doc:
        token._.is_custom = True
    return doc

nlp_custom = spacy.load("en_core_web_sm")

# Remove the NER component
if "ner" in nlp_custom.pipe_names:
    nlp_custom.remove_pipe("ner")
    print("Removed NER component")

# Add the custom component to the pipeline
nlp_custom.add_pipe("custom_component", name="custom_processor")

print(f"Custom pipeline: {nlp_custom.pipe_names}")

# Test the custom pipeline
text = "This is a test sentence."
doc = nlp_custom(text)

print(f"\nCustom attributes:")
for token in doc:
    print(f"'{token.text}': is_custom = {token._.is_custom}")


### Creating Custom Components

For more complex processing, you can create custom pipeline components:


In [ ]:
# Example: Custom sentiment analyzer component
from spacy.language import Language

@Language.component("sentiment_analyzer")
def sentiment_analyzer(doc):
    # Simple sentiment analysis based on word counts
    positive_words = ["good", "great", "excellent", "amazing", "wonderful", "fantastic"]
    negative_words = ["bad", "terrible", "awful", "horrible", "disgusting", "hate"]
    
    pos_count = sum(1 for token in doc if token.lemma_.lower() in positive_words)
    neg_count = sum(1 for token in doc if token.lemma_.lower() in negative_words)
    
    # Calculate sentiment score
    if pos_count + neg_count == 0:
        sentiment_score = 0.5  # Neutral
    else:
        sentiment_score = pos_count / (pos_count + neg_count)
    
    # Add sentiment to doc
    doc._.sentiment_score = sentiment_score
    doc._.sentiment_label = "positive" if sentiment_score > 0.6 else "negative" if sentiment_score < 0.4 else "neutral"
    
    return doc

# Register the sentiment attribute
from spacy.tokens import Doc
Doc.set_extension("sentiment_score", default=0.5)
Doc.set_extension("sentiment_label", default="neutral")

# Create pipeline with sentiment analysis
nlp_sentiment = spacy.load("en_core_web_sm")
nlp_sentiment.add_pipe("sentiment_analyzer", last=True)

# Test sentiment analysis
test_texts = [
    "This is a great movie!",
    "I hate this terrible film.",
    "The weather is okay today."
]

print("Sentiment Analysis:")
print("=" * 25)
for text in test_texts:
    doc = nlp_sentiment(text)
    print(f"'{text}' -> {doc._.sentiment_label} ({doc._.sentiment_score:.2f})")


## 4. Practice Exercises

### Exercise 1: Word Similarity Analysis
Create a function that finds the most similar words to a given word from a list of candidates.


In [ ]:
# Your solution here
def find_most_similar(target_word, candidates, nlp_model):
    """
    Find the most similar word from candidates to target_word
    """
    if target_word not in nlp_model.vocab:
        return None, 0.0
    
    target_token = nlp_model(target_word)
    similarities = []
    
    for candidate in candidates:
        if candidate in nlp_model.vocab:
            candidate_token = nlp_model(candidate)
            similarity = target_token.similarity(candidate_token)
            similarities.append((candidate, similarity))
    
    if not similarities:
        return None, 0.0
    
    # Sort by similarity (descending)
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[0]

# Test the function
target = "cat"
candidates = ["dog", "car", "house", "kitten", "computer"]
most_similar, score = find_most_similar(target, candidates, nlp_vectors)

print(f"Most similar to '{target}': '{most_similar}' (similarity: {score:.3f})")


### Exercise 2: Custom Pipeline Builder
Create a function that builds a custom spaCy pipeline with specified components.


In [ ]:
# Your solution here
def build_custom_pipeline(components_to_include=None, components_to_exclude=None):
    """
    Build a custom spaCy pipeline with specified components
    """
    # Start with a base model
    nlp_custom = spacy.load("en_core_web_sm")
    
    # Remove excluded components
    if components_to_exclude:
        for component in components_to_exclude:
            if component in nlp_custom.pipe_names:
                nlp_custom.remove_pipe(component)
                print(f"Removed: {component}")
    
    # Add included components (if not already present)
    if components_to_include:
        for component in components_to_include:
            if component not in nlp_custom.pipe_names:
                try:
                    nlp_custom.add_pipe(component)
                    print(f"Added: {component}")
                except Exception as e:
                    print(f"Could not add {component}: {e}")
    
    return nlp_custom

# Test the function
custom_nlp = build_custom_pipeline(
    components_to_exclude=["ner"],  # Remove NER
    components_to_include=["sentiment_analyzer"]  # Add sentiment (if available)
)

print(f"\nFinal pipeline: {custom_nlp.pipe_names}")


## 5. Summary and Key Takeaways

### What We Learned

1. **Word Vectors and Similarity**: Working with embeddings for semantic similarity
2. **Custom Tokenization**: Adding special cases and custom patterns
3. **Pipeline Customization**: Modifying spaCy's processing pipeline
4. **Custom Components**: Creating specialized processing components

### Key Concepts

- **Word Embeddings**: Numerical representations of words for similarity
- **Custom Tokenization**: Handling domain-specific text patterns
- **Pipeline Components**: Modular processing steps in spaCy
- **Custom Attributes**: Extending spaCy objects with new data

### Advanced Applications

- **Domain-Specific Processing**: Custom tokenization for scientific/medical text
- **Custom Analytics**: Adding sentiment analysis, custom metrics
- **Performance Optimization**: Minimal pipelines for specific tasks
- **Integration**: Combining spaCy with other NLP libraries

### Best Practices

1. **Use Appropriate Models**: Choose models with vectors for similarity tasks
2. **Test Custom Components**: Validate custom components thoroughly
3. **Pipeline Order**: Consider component dependencies and order
4. **Performance**: Remove unnecessary components for faster processing
5. **Documentation**: Document custom components and their purpose

### Next Steps

- Explore spaCy's training capabilities
- Learn about custom model training
- Integrate with machine learning frameworks
- Build production-ready NLP applications

### Common Pitfalls

1. **Vector Availability**: Not all models have word vectors
2. **Component Dependencies**: Some components require others
3. **Memory Usage**: Custom components may increase memory usage
4. **Performance Impact**: Adding components affects processing speed
